In [1]:
import pandas as pd
import numpy as np

In [2]:
am_ground = pd.read_csv("C:/Users/HELIOS-300/Desktop/WAVES/AM Full Code/Cameron_AM_Clean.csv")
am_ground.head()

C:\Users\HELIOS-300\AppData\Local\Temp\ipykernel_23672\498073532.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  am_ground = pd.read_csv("C:/Users/HELIOS-300/Desktop/WAVES/AM Full Code/Cameron_AM_Clean.csv")


,id,do_session,date_time,time,Modifier_1,Modifier_2,intensity_do,Comment,Activity_Type,posture_wbm,broad_domain,broad.behavior_do,posture_broad,broad.posture_do
0,1.0,DO1,2018-03-07 13:30:26,13:30:26,No movement,NaN,NaN,NaN,NaN,stand,NaN,NaN,stand_move,stationary
1,1.0,DO1,2018-03-07 13:30:27,13:30:27,No movement,NaN,NaN,NaN,NaN,stand,NaN,NaN,stand_move,stationary
2,1.0,DO1,2018-03-07 13:30:28,13:30:28,No movement,NaN,NaN,NaN,NaN,stand,NaN,NaN,stand_move,stationary
3,1.0,DO1,2018-03-07 13:30:29,13:30:29,No movement,NaN,NaN,NaN,NaN,stand,NaN,NaN,stand_move,stationary
4,1.0,DO1,2018-03-07 13:30:30,13:30:30,No movement,NaN,NaN,NaN,NaN,stand,NaN,NaN,stand_move,stationary


In [3]:
am_gt = pd.read_csv("C:/Users/HELIOS-300/Desktop/Data/am_gt_3.csv")
am_gt.head()

,day,id,actual_time,time,coding,primary_behavior,primary_posture,primary_upperbody,primary_intensity,num_postures,transition,posture_coding,broad_activity,detailed_activity,updated_activity,DO_session
0,7/24/2017,AM02,13:17:10,13H 17M 10S,non-sed,HA- housework,private/not coded,unknown,private/not coded,1,0,private/not coded,private/not coded,private/not coded,private/not coded,DO1
1,7/24/2017,AM02,13:17:11,13H 17M 11S,non-sed,HA- housework,private/not coded,unknown,private/not coded,1,0,private/not coded,private/not coded,private/not coded,private/not coded,DO1
2,7/24/2017,AM02,13:17:12,13H 17M 12S,non-sed,HA- housework,private/not coded,unknown,private/not coded,1,0,private/not coded,private/not coded,private/not coded,private/not coded,DO1
3,7/24/2017,AM02,13:17:13,13H 17M 13S,non-sed,HA- housework,private/not coded,unknown,private/not coded,1,0,private/not coded,private/not coded,private/not coded,private/not coded,DO1
4,7/24/2017,AM02,13:17:14,13H 17M 14S,non-sed,HA- housework,LA- stand and move with upper body movement,unknown,light,2,1,LA- stand and move light,mixed-activity,housework,mixed-activity,DO1


In [4]:
print("AM Groundtruth ID Unique:\n", am_ground["id"].unique(), "\n\nAM Groundtruth DO Unique:\n", am_ground["do_session"].unique())

AM Groundtruth ID Unique:
 [ 1.  2. nan  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. 20. 21. 22. 24. 25. 26. 27.] 

AM Groundtruth DO Unique:
 ['DO1' 'DO2' nan]


In [5]:
print("AM GT ID Unique:\n", am_gt["id"].unique(), "\n\nAM GT DO Unique:\n", am_gt["DO_session"].unique())

AM GT ID Unique:
 ['AM02' 'AM03' 'AM06' 'AM07' 'AM08' 'AM09' 'AM10' 'AM11' 'AM04' 'AM12'
 'AM05' 'AM13' 'AM14' 'AM01' 'AM16' 'AM15' 'AM17' 'AM18' 'AM19' 'AM20'
 'AM21' 'AM22' 'AM24' 'AM25' 'AM26' 'AM27'] 

AM GT DO Unique:
 ['DO1' 'DO2']


In [6]:
print("AM Ground Shape: ", am_ground.shape)
print("\nAM GT Shape: ", am_gt.shape)

AM Ground Shape:  (424022, 14)

AM GT Shape:  (332985, 16)


In [7]:
# Pair-level summary: duration, rows, gaps, duplicates, and mismatches

def normalize_am_ground_id(series):
    # map 1.0 -> AM01, 2.0 -> AM02, etc.
    return (
        series.dropna().astype(int).astype(str).str.zfill(2).radd("AM")
    )

am_ground_norm = am_ground.copy()
am_ground_norm["id_norm"] = pd.NA
am_ground_norm.loc[am_ground_norm["id"].notna(), "id_norm"] = normalize_am_ground_id(am_ground_norm["id"])

# align session column name
am_ground_norm = am_ground_norm.rename(columns={"do_session": "DO_session"})

# parse time columns
am_ground_norm["time_norm"] = pd.to_datetime(am_ground_norm["time"], format="%H:%M:%S", errors="coerce")
am_gt_norm = am_gt.copy()
am_gt_norm["time_norm"] = pd.to_datetime(am_gt_norm["actual_time"], format="%H:%M:%S", errors="coerce")


def summarize_pairs(df, id_col, session_col, time_col):
    use = df[[id_col, session_col, time_col]].dropna()
    use = use.sort_values([id_col, session_col, time_col])

    grouped = use.groupby([id_col, session_col])
    summary = grouped[time_col].agg(
        first_time="min",
        last_time="max",
        row_count="size",
        unique_time_count="nunique",
    ).reset_index()

    summary["duration_seconds"] = (
        (summary["last_time"] - summary["first_time"]).dt.total_seconds().astype("Int64")
    )
    summary["duration_hms"] = (
        pd.to_timedelta(summary["duration_seconds"], unit="s")
        .astype(str)
        .str.replace("0 days ", "", regex=False)
    )
    summary["expected_count"] = summary["duration_seconds"] + 1
    summary["gap_seconds"] = summary["expected_count"] - summary["unique_time_count"]
    summary["duplicate_rows"] = summary["row_count"] - summary["unique_time_count"]

    summary["first_time"] = summary["first_time"].dt.time
    summary["last_time"] = summary["last_time"].dt.time

    # count number of gap breaks (> 1 sec) using unique timestamps
    use_unique = use.drop_duplicates([id_col, session_col, time_col])
    diffs = use_unique.groupby([id_col, session_col])[time_col].diff().dt.total_seconds()
    gap_count = (
        diffs.gt(1)
        .groupby([use_unique[id_col], use_unique[session_col]])
        .sum()
        .rename("gap_count")
        .reset_index()
    )

    summary = summary.merge(
        gap_count,
        on=[id_col, session_col],
        how="left",
    )
    if "gap_count" not in summary.columns:
        summary["gap_count"] = 0
    summary["gap_count"] = summary["gap_count"].fillna(0).astype(int)

    # standardize id/session column names for downstream usage
    summary = summary.rename(columns={id_col: "id", session_col: "DO_session"})

    return summary.sort_values(["id", "DO_session"]).reset_index(drop=True)


am_ground_summary = summarize_pairs(
    am_ground_norm, id_col="id_norm", session_col="DO_session", time_col="time_norm"
)

am_gt_summary = summarize_pairs(
    am_gt_norm, id_col="id", session_col="DO_session", time_col="time_norm"
)

print("am_ground_summary (duration, rows, gaps, duplicates):")
print(am_ground_summary.to_string(index=False), "\n")
print("am_ground total rows across unique pairs:", am_ground_summary["row_count"].sum(), "\n")

print("am_gt_summary (duration, rows, gaps, duplicates):")
print(am_gt_summary.to_string(index=False), "\n")
print("am_gt total rows across unique pairs:", am_gt_summary["row_count"].sum(), "\n")

# pairs present in one dataset but not the other
pairs_ground = set(zip(am_ground_summary["id"], am_ground_summary["DO_session"]))
pairs_gt = set(zip(am_gt_summary["id"], am_gt_summary["DO_session"]))

only_in_ground = sorted(pairs_ground - pairs_gt)
only_in_gt = sorted(pairs_gt - pairs_ground)

print("Pairs only in am_ground:")
print(only_in_ground, "\n")
print("am_ground rows for pairs only in am_ground:",
      am_ground_summary[
          am_ground_summary[["id", "DO_session"]].apply(tuple, axis=1).isin(only_in_ground)
      ]["row_count"].sum(),
      "\n")

print("Pairs only in am_gt:")
print(only_in_gt)

am_ground_summary (duration, rows, gaps, duplicates):
  id DO_session first_time last_time  row_count  unique_time_count  duration_seconds duration_hms  expected_count  gap_seconds  duplicate_rows  gap_count
AM01        DO1   13:30:26  15:32:42       7337               7337              7336     02:02:16            7337            0               0          0
AM01        DO2   10:10:27  12:11:31       7265               7265              7264     02:01:04            7265            0               0          0
AM02        DO1   13:41:43  15:42:06       7224               7224              7223     02:00:23            7224            0               0          0
AM02        DO2   15:53:48  18:02:20      10840               7713              7712     02:08:32            7713            0            3127          0
AM03        DO1   18:24:00  20:24:14       7215               7215              7214     02:00:14            7215            0               0          0
AM03        DO2   13:4

In [8]:
test_df = am_ground[(am_ground["id"] == 15.0) & (am_ground["do_session"] == "DO2")]
test_df.iloc[20000:20020]

,id,do_session,date_time,time,Modifier_1,Modifier_2,intensity_do,Comment,Activity_Type,posture_wbm,broad_domain,broad.behavior_do,posture_broad,broad.posture_do
249099,15.0,DO2,2018-08-13 16:28:09,16:28:09,No movement,NaN,light,NaN,NaN,stand,NaN,NaN,stand_move,stationary
249100,15.0,DO2,2018-08-13 16:28:10,16:28:10,No movement,NaN,light,NaN,NaN,stand,NaN,NaN,stand_move,stationary
249101,15.0,DO2,2018-08-13 16:28:11,16:28:11,No movement,NaN,light,NaN,NaN,stand,NaN,NaN,stand_move,stationary
249102,15.0,DO2,2018-08-13 16:28:12,16:28:12,No movement,NaN,light,NaN,NaN,stand,NaN,NaN,stand_move,stationary
249103,15.0,DO2,2018-08-13 16:28:13,16:28:13,No movement,NaN,light,NaN,NaN,stand,NaN,NaN,stand_move,stationary
249104,15.0,DO2,2018-08-13 16:28:14,16:28:14,No movement,NaN,light,NaN,NaN,stand,NaN,NaN,stand_move,stationary
249105,15.0,DO2,2018-08-13 16:28:15,16:28:15,No movement,NaN,light,NaN,NaN,stand,NaN,NaN,stand_move,stationary
249106,15.0,DO2,2018-08-13 16:28:16,16:28:16,No movement,NaN,light,NaN,NaN,stand,NaN,NaN,stand_move,stationary
249107,15.0,DO2,2018-08-13 16:28:17,16:28:17,No movement,NaN,light,NaN,NaN,stand,NaN,NaN,stand_move,stationary
249108,15.0,DO2,2018-08-13 16:28:18,16:28:18,No movement,NaN,light,NaN,NaN,stand,NaN,NaN,stand_move,stationary
